# Lab 3: A Scoring and Case-Review Workflow

**Time:** about 75 minutes

You now have two kinds of score: an anomaly score that needs no labels (Lab 1) and a fraud probability learned from
confirmed cases (Lab 2). Neither is useful to the investigations team as a column in a spreadsheet. What they need
is a **workflow**: a clear, repeatable process that decides what happens to each payment, who looks at it, by when,
and why.

```
 payments ─► scores + business rules ─► combined risk ─► tier ─┬─ HOLD: stop payment, investigate now
                                                                ├─ REVIEW: investigate within 5 working days
                                                                ├─ QA SAMPLE: random check of low-risk payments
                                                                └─ PAY: normal processing
                                          reviewer decision ─► outcome recorded ─► feedback into the next model
```

In this lab you'll build that workflow with every threshold in a **configuration file**, so the investigations
manager can change how it behaves without touching code. You'll produce the queue with a plain-language reason
for every case, and measure how well the whole thing works.

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **workflow** | the fixed sequence of steps a payment goes through, from scoring to a recorded decision |
| **configuration file** | a plain-text file of settings (weights, thresholds) that people can change without editing code |
| **YAML** | a simple text format for configuration files: `name: value`, with indentation for groups |
| **business rule** | a fixed check written by people who know the job, e.g. "paid more than 130% of the entitlement" |
| **combined risk score** | one number per payment, mixing the model, the anomaly score and the rules |
| **tier** | the category a payment ends up in: HOLD, REVIEW, QA SAMPLE or PAY |
| **capacity** | how many cases the team can actually investigate in a month |
| **QA sample** | quality assurance: a small random check of payments the system considered safe |
| **out-of-fold prediction** | a score for each payment from a model that didn't see that payment during training |
| **KPI** | key performance indicator: a number managers use to judge how well something works |
| **audit log** | a permanent record of every decision: who, what, why and when |

## Where this lab fits

**What you've done so far.** In Lab 1 you ranked every 2025 payment by how unusual it is, with no labels, and saved the
scores. In Lab 2 you trained a fraud model on the confirmed cases, avoided two traps that flatter results, chose a
threshold from costs, and saved the model. On Day 2 you set a model's cut-off from the review team's capacity.

**What you'll do now.** You'll join those pieces into a working process: business rules alongside the two scores, one
combined risk score, tiers (HOLD, REVIEW, QA SAMPLE, PAY) that respect the team's monthly capacity, a queue with a
plain-language reason for every case, measures a manager can judge it by, a careful look at officer patterns, and an
audit log of every decision. All the settings live in a configuration file the investigations manager controls.

**Why it matters.** A score in a spreadsheet doesn't stop a single fraudulent payment. A clear process does: one that
tells the team which cases to open, in what order, why, and by when, and that records what was decided so the Fund
can defend every decision and learn from it.

**What happens next.** This completes Day 5. Day 6 moves from numbers to text: what members write to the Fund, and how
to read it at scale. The ideas from today (scores that rank, people who decide, a record of every decision) come back
in the deployed service and governance pack on Day 10.

### Theory: models, rules and people, each doing what it's good at

A fraud workflow has three kinds of decision-maker, and the design puts each where it's strongest:

| | Strength | Weakness | Role in this workflow |
|---|---|---|---|
| **Business rules** | transparent, immediate, encode what investigators know | rigid; noisy if written loosely | add points and give readable reasons |
| **Models** (supervised and anomaly) | weigh many signals at once; find patterns people miss | hard to explain; only as good as their data | rank the payments |
| **People** | judgement, context, fairness, accountability | slow and limited in number | make every final decision |

Two design choices follow. First, nothing is rejected automatically: the system can **hold** a payment, but only a
person can refuse one. Second, the team's **capacity** is part of the design, not an afterthought, because a queue
nobody can get through protects no one.

## Imports

New this time:

- `yaml`: reads and writes YAML configuration files;
- `json` and `datetime`: for writing the audit log in Section 8;
- `GroupKFold` and `cross_val_predict`: to score every payment fairly (Section 2).

`display.max_colwidth` lets pandas show long text (the reasons) without cutting it short.

In [ ]:
import os
import json
import warnings
from datetime import datetime
from pathlib import Path

# Skip a slow, noisy CPU-core check on Windows 11, and hide library notices.
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() - 1)
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yaml                                   # reads and writes YAML settings files
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import GroupKFold, cross_val_predict
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 80)     # show up to 80 characters of text per cell
SEED = 42

> **Where we are.** The libraries are loaded, including `yaml` for the settings file. Before any scoring, you'll write
> down every number a manager might want to change.

### Theory: configuration versus code

**Code** says *how* something is done: how to combine scores, how to assign tiers. **Configuration** says *with which
numbers*: how much each score counts, where the thresholds are, how many cases a month. Keeping them apart has big
advantages:

- The people who own the numbers (the Head of Investigations, finance) can read and change them without touching
  Python.
- A change to a threshold is a small, visible edit to one file, which can be reviewed and approved like any policy
  change.
- The configuration's version is recorded with every decision (Section 8), so you can always say which settings were
  in force when a payment was held.

A good test: if a business person might reasonably ask "can we try it with a different number?", that number belongs
in the configuration.

## 1. The configuration

Everything a manager might want to change lives here: weights, thresholds, capacity, and which rules are switched on.

Why keep it out of the code? Because the people who should decide these numbers (the Head of Investigations, finance)
aren't programmers, and every change should be visible and approved. A plain-text settings file can be read by anyone,
reviewed like any other document, and kept with a history of every change (in Git, the version-control tool from
Day 1).

It's written in **YAML**, a format designed to be easy for people to read:

- `name: value` sets a setting;
- indented lines belong to the line above (so `hold` and `review` belong to `thresholds`);
- `#` starts a comment;
- `{enabled: true, points: 1.0, ...}` puts a small group of settings on one line.

The settings, in plain words:

- **weights:** how much each of the three parts counts towards the combined score;
- **thresholds:** the combined score needed for HOLD and for REVIEW;
- **capacity:** how many cases the team can investigate per month;
- **qa_sample_rate:** the share of "safe" payments to check at random (0.005 = 0.5%);
- **rules:** each business rule, whether it's switched on, how many points it adds, and a description in plain words.

In [ ]:
CONFIG_TEXT = """
# Fraud review workflow configuration. Owner: Head of Investigations. Change via pull request only.
version: 1.0
weights:                 # how the combined risk score is built (should add up to 1)
  model_probability: 0.6
  anomaly_rank: 0.2
  rules: 0.2
thresholds:
  hold: 0.80             # combined score at or above this: hold the payment
  review: 0.45           # at or above this: review within 5 working days
capacity:
  reviews_per_month: 30  # what the team can actually investigate
qa_sample_rate: 0.005    # share of low-risk payments checked at random, to find blind spots
rules:
  shared_account_recent_change: {enabled: true, points: 1.0, description: "Account shared by 3+ members and changed in the last 180 days"}
  paid_above_entitlement:       {enabled: true, points: 1.0, description: "Paid more than 130% of entitlement (not an arrears payment)"}
  repeat_payment:               {enabled: true, points: 0.6, description: "Same amount, member and benefit paid again within 10 days"}
  single_officer_shared_account: {enabled: true, points: 0.8, description: "Shared account where every payment was approved by one officer"}
  fast_track_after_hours:       {enabled: true, points: 0.5, description: "Paid within 10 days of claim and approved after 17:00"}
"""

# Save it as a file, then read it back the way the real system would.
Path("review_config.yaml").write_text(CONFIG_TEXT.strip() + "\n")
CFG = yaml.safe_load(Path("review_config.yaml").read_text())      # safe_load: read settings only, never run code

# CFG is now an ordinary Python dictionary.
print(CFG["thresholds"])
print(CFG["capacity"])
print(CFG["rules"]["repeat_payment"])

> **Where we are.** The configuration is loaded into `CFG`: weights, thresholds, capacity, the QA sample rate and the
> five rules. Next, the three inputs the workflow combines.

## 2. Inputs: payments, scores and features

This notebook reuses the outputs of Labs 1 and 2 if they're in the folder, and rebuilds them quickly if they aren't,
so it can run on its own.

First the payments and the same 16 features as Labs 1 and 2 (Lab 1 explains each one):

In [ ]:
pay = pd.read_csv("benefit_payments_2025.csv", parse_dates=["payment_date"], dtype={"member_no": "string"})

# The same 16 features as Labs 1 and 2.
X = pd.DataFrame(index=pay.index)
X["log_amount"] = np.log1p(pay["amount_tzs"])
X["amount_vs_entitlement"] = pay["amount_tzs"] / pay["entitlement_tzs"]
X["log_days_since_account_change"] = np.log1p(pay["account_changed_days_ago"])
X["members_sharing_account"] = pay["members_sharing_account"]
X["payee_age"] = pay["payee_age"]
X["is_pension"] = pay["payment_type"].isin(["MONTHLY_PENSION", "ARREARS"]).astype(int)
X["months_since_verification"] = pay["months_since_life_verification"].fillna(0)
X["off_hours_approval"] = ((pay["approval_hour"] < 8) | (pay["approval_hour"] >= 17)).astype(int)
X["fast_payment"] = (pay["days_claim_to_payment"] < 10).astype(int)
X["mobile_money"] = (pay["account_type"] == "MOBILE_MONEY").astype(int)
X["officer_off_hours_rate"] = X["off_hours_approval"].groupby(pay["approving_officer"]).transform("mean")

# Repeats within 10 days (see Lab 1).
gap = pay.sort_values("payment_date").groupby(["member_no", "benefit_type", "amount_tzs"])["payment_date"].diff().dt.days
X["repeat_within_10d"] = (gap.reindex(pay.index) <= 10).astype(int)

X["type_arrears"] = (pay["payment_type"] == "ARREARS").astype(int)
X["type_lump_sum"] = (pay["payment_type"] == "LUMP_SUM").astype(int)
X["type_grant"] = (pay["payment_type"] == "GRANT").astype(int)
X["type_instalment"] = (pay["payment_type"] == "INSTALMENT").astype(int)
print(X.shape)

**Score 1: the fraud probability.** Score every payment with a model that never saw that member: these are called
**out-of-fold predictions**. (`fraud_model.joblib` from Lab 2 is trained on all of 2025; it's the one you'd use on 2026
payments. Using it here would mean scoring payments it learned from.)

**Why out-of-fold?** If you scored 2025 payments with a model trained on those same payments, it would recognise
them and look almost perfect, and every KPI (key performance indicator) later in this notebook would be flattering
nonsense.

`cross_val_predict` with `GroupKFold` fixes that:

1. split the members into five groups;
2. train on four groups and score the payments of the fifth;
3. repeat five times, so every group gets scored once.

Every payment ends up scored by a model that never saw that member, which is the closest you can get to "how would
this have done on new payments?" without waiting a year. `method="predict_proba"` asks for probabilities rather than
yes/no answers; column 1 is the probability of fraud.

In [ ]:
fraud_weight = (pay["confirmed_fraud"] == 0).sum() / (pay["confirmed_fraud"] == 1).sum()     # as in Lab 2
fraud_model = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                            scale_pos_weight=fraud_weight, eval_metric="logloss", random_state=SEED, n_jobs=4)

# GroupKFold: 5 groups of members. Train on 4, score the 5th, five times over.
# cross_val_predict collects the scores, so every payment gets one from a model that never saw its member.
out_of_fold = cross_val_predict(fraud_model, X, pay["confirmed_fraud"], groups=pay["member_no"],
                                cv=GroupKFold(n_splits=5), method="predict_proba")
pay["model_probability"] = out_of_fold[:, 1]
pay["model_probability"].describe()

**Score 2: the anomaly rank** from Lab 1 (the ensemble score, already a percentile between 0 and 1). If Lab 1 hasn't
been run, the `else` part fits a quick isolation forest and turns its scores into percentiles instead.

`merge` joins two tables on a shared column, here `payment_id`, so each payment picks up its anomaly score.
`how="left"` keeps every payment, even if one had no score.

In [ ]:
if Path("anomaly_scores.csv").exists():
    anomaly = pd.read_csv("anomaly_scores.csv")[["payment_id", "ensemble_score"]]
    pay = pay.merge(anomaly, on="payment_id", how="left")      # attach Lab 1's score to each payment
    pay["anomaly_rank"] = pay["ensemble_score"]
    print("using the anomaly scores from Lab 1")
else:
    iso = IsolationForest(n_estimators=300, random_state=SEED, n_jobs=-1)
    iso.fit(X)
    iso_scores = pd.Series(-iso.score_samples(X), index=pay.index)
    pay["anomaly_rank"] = iso_scores.rank(pct=True)            # percentile, 0 to 1
    print("Lab 1 scores not found: computed a quick isolation forest instead")

pay[["model_probability", "anomaly_rank"]].describe().round(3)

> **Where we are.** Every payment now has two scores: a fraud probability from a model that never saw its member, and an
> anomaly rank from Lab 1. The third input is what investigators already know.

## 3. Business rules, each with a reason a person can read

Rules capture what investigators already know. They're transparent, auditable, and they catch known patterns the
moment they appear, even before a model has seen enough examples to learn them.

The five rules, in plain words (the descriptions in the configuration say the same):

1. **shared account, recent change:** 3 or more members paid into one account, which was changed less than 180 days ago;
2. **paid above entitlement:** more than 130% of what the member is owed (arrears are excluded, since back-pay is
   legitimately larger);
3. **repeat payment:** the same amount to the same member for the same benefit within 10 days;
4. **single officer, shared account:** a shared account where every payment was approved by the same officer;
5. **fast track after hours:** paid within 10 days of the claim *and* approved after 17:00.

Each rule becomes a True/False column: True where the rule fires. `&` means "and". For rule 4,
`transform("nunique")` counts how many different officers approved payments into each account and writes that count on
every payment, as in Lab 1.

The summary table shows how often each rule fires, and how many of those payments were confirmed fraud.

In [ ]:
# How many different officers approved payments into each account? (one number per payment)
officers_per_account = pay.groupby("payee_account")["approving_officer"].transform("nunique")

rule_hits = pd.DataFrame(index=pay.index)
rule_hits["shared_account_recent_change"] = (pay["members_sharing_account"] >= 3) & (pay["account_changed_days_ago"] < 180)
rule_hits["paid_above_entitlement"] = (pay["amount_tzs"] > 1.3 * pay["entitlement_tzs"]) & (pay["payment_type"] != "ARREARS")
rule_hits["repeat_payment"] = X["repeat_within_10d"] == 1
rule_hits["single_officer_shared_account"] = (pay["members_sharing_account"] >= 3) & (officers_per_account == 1)
rule_hits["fast_track_after_hours"] = (X["fast_payment"] == 1) & (X["off_hours_approval"] == 1)

# For each rule: how often it fires, and how many of those were confirmed fraud.
rule_summary = pd.DataFrame({
    "payments hit": rule_hits.sum(),
    "of which confirmed fraud": rule_hits[pay["confirmed_fraud"] == 1].sum(),
})
rule_summary

Now turn the rules into **points** and **reasons**, using the configuration. For each rule:

- skip it if it's switched off (`enabled: false`); `continue` jumps straight to the next rule;
- add its points to every payment it fires on;
- add its description to those payments' reason text, followed by `"; "` so several reasons can be listed together.

`pay.loc[fires, "column"]` means "the rows where `fires` is True, in that column".

In [ ]:
# Add up the points of every enabled rule that fires, and collect each rule's description as the reason.
pay["rule_points"] = 0.0
pay["rule_reasons"] = ""

for rule_name in rule_hits.columns:
    rule = CFG["rules"][rule_name]          # this rule's settings from the configuration
    if not rule["enabled"]:
        continue                            # switched off: skip it
    fires = rule_hits[rule_name]
    pay.loc[fires, "rule_points"] = pay.loc[fires, "rule_points"] + rule["points"]
    pay.loc[fires, "rule_reasons"] = pay.loc[fires, "rule_reasons"] + rule["description"] + "; "

# A few payments where at least one rule fired.
pay.loc[pay["rule_points"] > 0, ["payment_id", "rule_points", "rule_reasons"]].head()

Look at the second column of the rule summary. Some rules are sharp, and some fire mostly on legitimate payments. A
rule that's nearly always wrong wastes investigators' time and still deserves a second look: maybe it needs tightening,
maybe it should add points to the score without forcing a review on its own. That's exactly why rules add *points*
here instead of sending cases straight to the queue.

> **Where we are.** Every payment has rule points and a list of reasons for any rule that fired, and you've seen which
> rules are sharp and which are noisy. Now the three signals become one score.

### Theory: combining scores on different scales

You can only average numbers that mean comparable things. A probability of 0.9, an anomaly percentile of 0.95 and
"2.4 rule points" are on different scales, so they're first each turned into a number between 0 and 1. Then a
**weighted average** combines them:

```
risk = 0.6 × model probability + 0.2 × anomaly rank + 0.2 × (rule points ÷ maximum points)
```

The weights express how much the Fund trusts each signal. They're a judgement, not a law of nature, which is why
they're in the configuration. The model gets the most weight because it was learned from confirmed cases and tested
honestly; the anomaly and rule parts still let a payment rise up the list for reasons the model can't see.

**Tiers** then turn the single score into an action, using two thresholds. This is the same idea as Day 2's action
bands, but now the bands come from a file the manager controls.

## 4. One combined score, then tiers

You now have three signals per payment, and they're on different scales: a probability (0 to 1), a percentile (0 to 1),
and rule points (0 up to the total of all rules' points). To combine them:

1. turn the rule points into a share of the maximum possible, so they're between 0 and 1 too. `clip(upper=1)` makes
   sure nothing goes above 1;
2. take a **weighted average** with the weights from the configuration: 60% model, 20% anomaly, 20% rules.

For example, a payment with model probability 0.9, anomaly rank 0.95 and half the maximum rule points scores
0.6 × 0.9 + 0.2 × 0.95 + 0.2 × 0.5 = 0.83.

In [ ]:
weights = CFG["weights"]

# The most points a payment could get, if every enabled rule fired.
max_points = 0
for rule in CFG["rules"].values():
    if rule["enabled"]:
        max_points = max_points + rule["points"]

# Rule points as a share of the maximum, between 0 and 1.
rules_share = (pay["rule_points"] / max_points).clip(upper=1)

# The combined score: a weighted average of the three parts.
pay["risk_score"] = (weights["model_probability"] * pay["model_probability"]
                     + weights["anomaly_rank"] * pay["anomaly_rank"]
                     + weights["rules"] * rules_share).round(4)

pay["risk_score"].describe()

Now put each payment in a tier using the two thresholds from the configuration. Start everyone at PAY, then move up
those at or above the REVIEW threshold, then move up again those at or above the HOLD threshold. The order matters:
a payment above 0.80 is first set to REVIEW, then overwritten with HOLD.

In [ ]:
# Tiers from the thresholds in the configuration. Start with PAY, then upgrade.
thresholds = CFG["thresholds"]
pay["tier"] = "PAY"
pay.loc[pay["risk_score"] >= thresholds["review"], "tier"] = "REVIEW"
pay.loc[pay["risk_score"] >= thresholds["hold"], "tier"] = "HOLD"

pay["tier"].value_counts()

### Respect the team's capacity

A queue the team can't get through is no better than no queue: cases just pile up. The configuration says the team can
investigate 30 cases a month. HOLD cases always go to the team. REVIEW cases fill whatever capacity is left, highest
score first. Anything that doesn't fit is paid normally but kept on a watch-list, and the manager sees the overflow
count. If it's large every month, that's the evidence for more staff.

The loop goes through the months one at a time. For each month:

1. count the HOLD cases; the room left is the capacity minus that (never below 0, hence `max(0, ...)`);
2. sort that month's REVIEW cases by score, highest first;
3. the first `room` of them stay as REVIEW; the rest (`reviews.index[room:]`, "from position `room` to the end")
   become "PAY (watch-list)".

In [ ]:
pay["month"] = pay["payment_date"].dt.to_period("M")       # e.g. 2025-03
capacity = CFG["capacity"]["reviews_per_month"]
overflow = {}       # month -> number of REVIEW cases that didn't fit

for month in sorted(pay["month"].unique()):
    this_month = pay[pay["month"] == month]

    # HOLD cases always go to the team; the room left over is for REVIEW cases.
    n_holds = (this_month["tier"] == "HOLD").sum()
    room = max(0, capacity - n_holds)

    # This month's REVIEW cases, highest score first. The first `room` of them fit; the rest don't.
    reviews = this_month[this_month["tier"] == "REVIEW"].sort_values("risk_score", ascending=False)
    do_not_fit = reviews.index[room:]

    pay.loc[do_not_fit, "tier"] = "PAY (watch-list)"
    overflow[str(month)] = len(do_not_fit)

# One row: how many reviews didn't fit, per month.
pd.Series(overflow, name="reviews that didn't fit").to_frame().T

### The random quality check

Finally, choose a random 0.5% of the payments that are being paid normally and mark them for a routine check.
`np.random.default_rng(SEED)` creates a random number generator with a fixed seed (so the same payments are picked
each run), and `rng.choice(..., replace=False)` picks that many different payments.

In [ ]:
# A random quality-assurance sample from everything that is being paid normally.
rng = np.random.default_rng(SEED)
being_paid = pay.index[pay["tier"].isin(["PAY", "PAY (watch-list)"])]
n_sample = int(CFG["qa_sample_rate"] * len(being_paid))          # 0.5% of them
qa_rows = rng.choice(being_paid, n_sample, replace=False)        # pick them at random, no repeats
pay.loc[qa_rows, "tier"] = "QA SAMPLE"

pay["tier"].value_counts()

Why the **QA sample**? Without it, the team only ever looks at what the system flags, so it can never discover what
the system misses. A small random check of "clean" payments is how you measure your blind spots, and it's the only
source of labels for fraud patterns nobody has thought of yet.

> **Where we are.** Every payment has a combined score and a tier; the REVIEW tier has been trimmed to the team's monthly
> capacity, with the overflow on a watch-list; and a random QA sample of "safe" payments has been picked. Now turn that
> into something an investigator can work from.

### Theory: why every case needs a reason

A case that says only "risk score 0.87" leaves the investigator with nowhere to start, and leaves the Fund unable to
explain itself if a member complains. A good reason:

- **uses the payment's own facts** ("3 members paid into this account, changed 40 days ago"), not model jargon;
- **points to what to check** (the account change form, the life verification);
- **never accuses** ("resembles confirmed fraud cases", not "fraudster").

Rules are naturally explainable, which is one of the reasons they're kept alongside the models. For the model and
anomaly parts, the reason says which signal raised the case and how strongly.

## 5. The queue, as an investigator would see it

Every case gets a reason written in plain language, a recommended action and a due date. "Risk score 0.87" on its
own isn't something an officer can act on or explain to a member.

The reason is built up in pieces:

1. start with the descriptions of the rules that fired;
2. if the model probability is 0.5 or more, add "Resembles confirmed fraud cases" with the probability;
3. if the anomaly rank is 0.99 or more, add "Among the 1% most unusual payments this year";
4. if nothing was added, use a general reason;
5. `str.rstrip("; ")` tidies the end by removing the last `"; "`.

In [ ]:
# The cases the team will work on.
work = pay[pay["tier"].isin(["HOLD", "REVIEW", "QA SAMPLE"])].copy()

# Start the reason with the rules that fired, then add the scores where they're high.
work["reason"] = work["rule_reasons"]

high_model = work["model_probability"] >= 0.5
work.loc[high_model, "reason"] = (work.loc[high_model, "reason"] + "Resembles confirmed fraud cases (model "
                                  + work.loc[high_model, "model_probability"].round(2).astype(str) + "); ")

very_unusual = work["anomaly_rank"] >= 0.99
work.loc[very_unusual, "reason"] = work.loc[very_unusual, "reason"] + "Among the 1% most unusual payments this year; "

# Cases with no specific reason.
work.loc[work["reason"] == "", "reason"] = "Combined score above review threshold"
work["reason"] = work["reason"].str.rstrip("; ")        # remove the trailing "; "

work[["payment_id", "tier", "reason"]].head()

Next, the recommended action for each tier (`map` looks up each tier in the `actions` dictionary) and a due date.
`pd.offsets.BDay(5)` means "5 business days later", skipping weekends: HOLD cases are due in 1 working day, the others
in 5.

In [ ]:
# What the investigator should do, by tier.
actions = {
    "HOLD": "Stop payment; verify member identity and account ownership before release",
    "REVIEW": "Investigate; payment may proceed",
    "QA SAMPLE": "Routine check of documents and account ownership",
}
work["action"] = work["tier"].map(actions)

# Due date: 1 working day for HOLD, 5 for the others. BDay = business day (weekends skipped).
work["due"] = work["payment_date"] + pd.offsets.BDay(5)
is_hold = work["tier"] == "HOLD"
work.loc[is_hold, "due"] = work.loc[is_hold, "payment_date"] + pd.offsets.BDay(1)

Put the columns an investigator needs in a sensible order and sort the queue: by tier (alphabetically, which happens
to put HOLD first, then QA SAMPLE, then REVIEW), and within each tier by risk score, highest first.

In [ ]:
# The queue: HOLD first, then QA SAMPLE, then REVIEW; highest score first within each.
columns = ["payment_id", "payment_date", "tier", "risk_score", "member_no", "benefit_type", "amount_tzs",
           "reason", "action", "due"]
queue_view = work.sort_values(["tier", "risk_score"], ascending=[True, False])[columns]
queue_view.head(12)

Save the queue as a CSV file in an `outputs` folder, which the team could open in Excel. `exist_ok=True` means "don't
complain if the folder is already there".

In [ ]:
Path("outputs").mkdir(exist_ok=True)
queue_view.to_csv("outputs/review_queue_2025.csv", index=False)
print(f"{len(queue_view):,} cases written to outputs/review_queue_2025.csv")

> **Where we are.** The queue is saved, with a reason, an action and a due date for every case. The last questions are
> the ones a manager asks: is it working, and is each part pulling its weight?

### Theory: measuring a process, not just a model

On Days 2 and 5 you measured models with precision, recall and PR-AUC. A manager judges a **process** with measures in
their own language:

| Measure | Question it answers |
|---|---|
| **hit rate** per tier | is the team's time well spent? |
| **share of all fraud** caught | how much of the problem does the process find? |
| **value of fraud** stopped, in TZS | what is it worth? |
| **fraud in PAY** | what does it miss? |
| **watch-list hit rate** | what would one more investigator catch? |
| **rule effectiveness** | which rules earn their place? |

These are the **KPIs** (key performance indicators) to review every quarter. Here `confirmed_fraud` stands in for the
investigators' findings; in real use these numbers fill in as cases are closed.

## 6. How well does the workflow work?

In real life you'd wait for investigators' findings. Here, `confirmed_fraud` stands in for them. Look at the workflow
the way a manager would: per tier, and in shillings.

To put a value on fraud, use the same rough rule as Lab 2: stopping a fraudulent monthly pension saves 12 months of
payments; stopping any other fraudulent payment saves the amount itself. Multiplying by `confirmed_fraud` (0 or 1)
gives 0 for payments that weren't fraud.

Then `groupby("tier").agg(...)` builds one row per tier, with:

- **payments:** how many payments are in the tier;
- **confirmed_fraud:** how many of those were fraud;
- **hit_rate:** the share that were fraud;
- **fraud_value:** the value of that fraud, in TZS millions;
- **share_of_all_fraud:** the tier's share of all the fraud there is.

In [ ]:
# Rough value of stopping a fraudulent payment: 12 months for a monthly pension, the amount itself otherwise.
months = np.where(pay["payment_type"] == "MONTHLY_PENSION", 12, 1)
pay["fraud_value"] = months * pay["amount_tzs"] * pay["confirmed_fraud"]     # 0 for non-fraud payments

kpi = pay.groupby("tier").agg(
    payments=("payment_id", "count"),
    confirmed_fraud=("confirmed_fraud", "sum"),
    hit_rate=("confirmed_fraud", "mean"),
    fraud_value_tzs=("fraud_value", "sum"),
)
kpi["fraud_value_tzs_m"] = kpi["fraud_value_tzs"] / 1e6                         # in millions
kpi["share_of_all_fraud"] = kpi["confirmed_fraud"] / pay["confirmed_fraud"].sum()
kpi.drop(columns="fraud_value_tzs").round(3)

Read each row:

- **HOLD and REVIEW:** what share of cases were real (the hit rate), and what share of all fraud they caught. A high hit
  rate means the team's time is well spent. A high share of all fraud means members' money is protected.
- **PAY:** any fraud here was missed. This is the number that should keep you honest.
- **PAY (watch-list):** cases that scored high enough for review but didn't fit the team's capacity. Look at their hit
  rate. If it's high, and here it is, that row is the business case for another investigator, written in shillings.
- **QA SAMPLE:** a random slice of "clean" payments. If it keeps finding fraud, the model and rules have a blind spot.

### Which rules earn their place?

A rule is worth keeping if it's accurate, *or* if it catches fraud the model would have missed. For each rule, three
numbers:

- **fired:** how many payments it fired on;
- **precision:** of those, the share that were fraud;
- **fraud caught only by this rule:** fraud payments the rule fired on that the model scored below 0.5, so the model
  alone would probably have let them through.

In [ ]:
is_fraud = pay["confirmed_fraud"] == 1
model_missed = pay["model_probability"] < 0.5

rule_effectiveness = pd.DataFrame({
    # how many payments each rule fired on
    "fired": rule_hits.sum(),
    # of those, the share that were fraud
    "precision": rule_hits[is_fraud].sum() / rule_hits.sum(),
    # fraud the rule caught that the model scored below 0.5
    "fraud caught only by this rule": rule_hits[is_fraud & model_missed].sum(),
})
rule_effectiveness.round(3)

A rule with low precision that also catches nothing the model misses is a candidate for switching off. A rule with
low precision that catches fraud the model can't see earns its place, even though it's noisy.

> **Where we are.** You know how each tier performs, what's being missed, what extra capacity would buy, and which rules
> earn their place. One more kind of pattern needs looking at, with care.

## 7. Patterns by approving officer, handled carefully

Several fraud schemes need someone on the inside. Looking at which officers approved flagged cases is legitimate
and important, and it's also sensitive. The output is a **prompt for a supervisor's review of controls**, never an
accusation, and it goes to the right people only.

For each officer: how many payments they approved, how many were flagged (HOLD or REVIEW), and what share were
approved after 17:00. The mean of a True/False column is the share of True.

In [ ]:
pay["flagged"] = pay["tier"].isin(["HOLD", "REVIEW"])
pay["after_hours"] = pay["approval_hour"] >= 17

by_officer = pay.groupby("approving_officer").agg(
    payments=("payment_id", "count"),
    flagged=("flagged", "sum"),
    after_hours=("after_hours", "mean"),       # share approved after 17:00
)
by_officer["flagged_share"] = by_officer["flagged"] / by_officer["payments"]
by_officer.sort_values("flagged_share", ascending=False).head(8).round(3)

If one officer's approvals are flagged far more often than colleagues', the proportionate response is a control
review: are approvals being checked by a second officer (the "four-eyes" principle)? Is one person handling both
account changes and payments (a segregation-of-duties problem)? A person, not a model, decides what happens next.

> **Where we are.** The workflow scores, tiers, explains, measures itself and highlights control questions about
> officers. What's still missing is a record of what people actually decide.

### Theory: the feedback loop

Every decision an investigator records is a **label**: confirmed fraud, or cleared. Those labels are next year's
training data. So the workflow is a loop, not a line:

```
score ─► queue ─► investigate ─► record outcome ─► retrain with new labels ─► score ...
```

Two things make the loop healthy:

- **Cleared cases matter as much as confirmed ones.** They teach the model what innocent-but-unusual looks like.
- **The QA sample keeps the loop honest.** If the team only ever investigates what the system flags, the labels only
  ever confirm the system's own beliefs, and its blind spots never get labelled. Random checks of "safe" payments are
  the only way to find what it misses.

An **append-only** log (lines are added, never changed) makes the record trustworthy: nobody can quietly rewrite a past
decision.

## 8. Closing the loop

A workflow isn't finished when the queue is produced. Record what happens to every case, because those records are
next year's training labels and next year's evidence that the process is fair.

Each decision is added as one line of JSON to a log file (a format called JSON Lines). JSON is a plain-text way of
writing a dictionary, e.g. `{"payment_id": "P123", "outcome": "CLEARED"}`. Lines are only ever added, never changed or
deleted, so the log is a trustworthy history.

The function below writes one decision. Opening the file with `"a"` (append) adds to the end without touching what's
already there; `"w"` would wipe it.

In [ ]:
AUDIT_LOG = Path("outputs/review_decisions.jsonl")


def record_decision(payment_id, reviewer, outcome, notes):
    """Append one decision to the audit log: who, what, why, when, and under which configuration version."""
    entry = {
        "payment_id": payment_id,
        "reviewer": reviewer,
        "outcome": outcome,
        "notes": notes,
        "config_version": CFG["version"],                              # which settings were in force
        "decided_at": datetime.now().isoformat(timespec="seconds"),    # e.g. 2026-03-14T10:22:05
    }
    # "a" = append: add to the end of the file, never overwrite.
    with open(AUDIT_LOG, "a", encoding="utf-8") as log_file:
        log_file.write(json.dumps(entry) + "\n")                       # one line per decision
    return entry

Record two example decisions, one confirmed fraud and one cleared, for the first two cases in the queue, then print the
log.

In [ ]:
first_case = queue_view.iloc[0]["payment_id"]
second_case = queue_view.iloc[1]["payment_id"]

record_decision(first_case, "INV-014", "CONFIRMED_FRAUD",
                "Account holder is not the member; account opened with a third party's ID. Referred for legal action.")
record_decision(second_case, "INV-022", "CLEARED",
                "Member verified in person at branch; account confirmed in own name. Payment released.")

print(AUDIT_LOG.read_text())

Every decision records who made it, when, why, and under which configuration version. That answers the questions
an auditor, a court or an aggrieved member will eventually ask.

Some design rules for the human side of the workflow, worth agreeing before go-live:

1. **No automatic rejection.** The system can hold a payment. Only a person can refuse one.
2. **Tell the member.** A held payment gets an SMS within one working day saying what's needed. It should never say
   "suspected fraud".
3. **Four eyes on outcomes.** A second officer confirms any finding of fraud before it's acted on.
4. **Time limits.** A held payment is released automatically, with a note on the file, if nobody acts within the
   agreed period. Members shouldn't bear the cost of a backlog.
5. **Feedback.** Cleared cases are labelled as *not fraud*. They're as valuable for retraining as confirmed ones.
6. **Review the configuration** quarterly with the investigations manager, using the KPI and rule-effectiveness
   tables from Section 6.

---

## Looking back, looking ahead

**What you did in this lab.** You built a complete fraud review workflow: a configuration file owned by the business,
two scores and five business rules combined into one risk score, tiers that respect the team's capacity, a random QA
sample, a queue with plain-language reasons, KPIs and rule-effectiveness tables, a careful look at officer patterns,
and an append-only audit log.

**What you did today.** Day 5 took fraud detection from start to finish: finding the unusual with no labels (Lab 1),
learning from rare confirmed cases without fooling yourself (Lab 2), and turning both into a process a small team can
work (Lab 3).

**The ideas to carry forward.**

- Unusual isn't fraudulent; models rank, people decide.
- Settings belong in configuration, owned by the business.
- Capacity is part of the design.
- Every case needs a reason in plain language.
- Record every decision; the records are tomorrow's labels and today's accountability.

**Tomorrow (Day 6).** From numbers to words: sentiment, complaint routing and entity extraction on what members write
to the Fund.

## Your turn

1. **Change the dials.** The investigations manager wants to try more weight on the rules (`rules: 0.4`,
   `model_probability: 0.4`) and a capacity of 20 a month. Edit `CONFIG_TEXT`, rerun from Section 1, and compare the
   KPI table. What did you gain and what did you lose?
2. **Add a rule.** Pensioners aged over 85 whose account changed in the last 90 days are a known ghost-pension pattern
   elsewhere. Add it as a rule (`elderly_recent_account_change`, 0.7 points), with a description, and check its
   precision.
3. **Write the SMS.** Draft the text message a member receives when their payment is held, in English and in
   Swahili, in under 160 characters each. It must not accuse, it must say what to do, and it must say how long it will
   take.

In [ ]:
# Your code here

## Suggested answer to (2)

The rule is one True/False condition. Check how often it fires and how often it's right before adding it to the
configuration:

In [ ]:
elderly_recent_change = (pay["payee_age"] > 85) & (pay["account_changed_days_ago"] < 90)

fired = elderly_recent_change.sum()                     # payments it fires on
caught = (elderly_recent_change & is_fraud).sum()       # of those, confirmed fraud
print("fires on", fired, "payments")
print("precision:", round(caught / fired, 2))
print("catches", caught, "confirmed fraud payments")

For (3), one possibility. English: *"NSSF: Your May pension is on hold pending a routine check. Please visit any branch
with your NIDA card within 14 days. Call 0800 xxx xxx for help."* Swahili: *"NSSF: Pensheni yako ya Mei imesitishwa
kwa uhakiki wa kawaida. Tafadhali fika tawi lolote na kitambulisho cha NIDA ndani ya siku 14."*